# Phase 1: Document Vectorization

This notebook demonstrates the first stage of the RAG pipeline: **Ingestion and Storage**.

We will cover:
1. **Extraction**: Reading text from a PDF.
2. **Chunking**: Splitting text into manageable segments.
3. **Embedding**: Converting text into semantic vectors.
4. **Indexing**: Saving segments to a local ChromaDB store.

In [1]:
import os
import sys

# Add the src directory to the python path so we can import our modules
sys.path.append(os.path.abspath('../src'))

from ingest.parser import extract_text_from_pdf
from ingest.chunker import chunk_text
from rag.retriever import VectorRetriever

print("Modules loaded successfully.")

d:\Engineering Works\pdf-knowledge-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modules loaded successfully.


## 1. Text Extraction

We'll use `pdfplumber` (wrapped in our `parser` module) to extract raw text from a sample PDF.

> **Note**: Our improved parser now understands **two-column layouts** and fixes **character spacing**!

In [2]:
pdf_path = "../data/raw/sample.pdf"

if not os.path.exists(pdf_path):
    print(f"Error: {pdf_path} not found. Please ensure a sample PDF exists in data/raw/")
else:
    text = extract_text_from_pdf(pdf_path)
    print(f"Extracted {len(text)} characters from the PDF.")
    print("\n--- Sample Text (First 1000 chars) ---")
    print(text[:1000])

Extracted 73732 characters from the PDF.

--- Sample Text (First 1000 chars) ---
Capturing Monetarily Ex
Smart Contracts via Audi
Fuzz
Bowen Cai1, Weiheng Bai1, Hangyu
1University of Minnesota - Twin Cities, 2Fu
{cai000254, bai00093}@umn.edu,
{yolu6176}@uni.sydney.e
Abstract—Smartcontractshaveextendedblockchainfunctional-
itybeyondsimpletransactions,poweringcomplexapplicationslike
decentralized finance (DeFi). However, this complexity introduces
serious security challenges, including price manipulation and
inflation attacks. Despite the development of various security
tools, the rapid rise in financially motivated exploits continues to
pose a significant threat to the blockchain ecosystem.
ThesefinanciallymotivatedexploitsoftenstemfromMonetarily
ExploitableVulnerabilities(MEVuls),whichrefertovulnerabilities
arisingfromexploitableimplementationsinmonetarytransactions
orvalue-transferlogic.Duetotheircomplexity,intricatechainsof
functioncalls,multifacetedlogic,anddiversemanifestationsacro

## 2. Text Chunking

Raw text is too long for LLMs to process efficiently. We split it into smaller overlapping chunks.

In [5]:
chunk_size = 500
chunk_overlap = 100
chunks = chunk_text(text, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
print(f"Split text into {len(chunks)} chunks.")

print("\n--- Chunk 1 ---")
print(chunks[0])

print("\n--- Chunk 2 ---")
print(chunks[1])

print("\n--- Chunk 3 ---")
print(chunks[2])

Split text into 182 chunks.

--- Chunk 1 ---
Capturing Monetarily Ex
Smart Contracts via Audi
Fuzz
Bowen Cai1, Weiheng Bai1, Hangyu
1University of Minnesota - Twin Cities, 2Fu
{cai000254, bai00093}@umn.edu,
{yolu6176}@uni.sydney.e
Abstract—Smartcontractshaveextendedblockchainfunctional-
itybeyondsimpletransactions,poweringcomplexapplicationslike
decentralized finance (DeFi). However, this complexity introduces
serious security challenges, including price manipulation and
inflation attacks. Despite the development of various security

--- Chunk 2 ---
inflation attacks. Despite the development of various security
tools, the rapid rise in financially motivated exploits continues to
pose a significant threat to the blockchain ecosystem.
ThesefinanciallymotivatedexploitsoftenstemfromMonetarily
ExploitableVulnerabilities(MEVuls),whichrefertovulnerabilities
arisingfromexploitableimplementationsinmonetarytransactions
orvalue-transferlogic.Duetotheircomplexity,intricatechainsof
functioncalls,mu

## 3. Vector Storage (ChromaDB)

Now we convert these chunks into embeddings using a local `sentence-transformer` model and save them to our local ChromaDB.

In [6]:
retriever = VectorRetriever()

# Clear existing items to avoid duplicates during testing
try:
    retriever.client.delete_collection(retriever.collection.name)
    retriever.collection = retriever.client.get_or_create_collection(name=retriever.collection.name)
    print("Cleaned existing collection.")
except:
    pass

# Add chunks to store
metadatas = [{"source": "sample.pdf"} for _ in chunks]
retriever.add_chunks(chunks, metadatas=metadatas)

print(f"Successfully indexed {retriever.get_collection_count()} items in the vector store.")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Cleaned existing collection.


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Successfully indexed 182 items in the vector store.


## 4. Verification

Let's check if we can retrieve something back (even without full search logic implemented yet).

In [ ]:
results = retriever.collection.peek(limit=5)
print(f"Retrieved {len(results['ids'])} items from the store.")

type(results)

dict